# Geo-outperformer (d = 2, 3, 4 or 5)

Standalone backward stochastic-policy-gradient (SPG) experiments under uncertain volatility and correlation. This notebook contains all required implementation code. It does not read or write local files.

Change only `d` in the configuration cell to select the number of assets (2, 3, 4 or 5). The payoff, correlation representation, neural-network dimensions and dual optimizer adapt to `d`.

**Workflow:** classical SPG training and independent Monte Carlo pricing, Sobolev SPG training and independent pricing, then separate approximate dual evaluations from both learned critic gradients. 

## Dependencies and imports

Requires PyTorch and Matplotlib. No local Python modules are needed.

In [ ]:
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Networks and controlled log-Euler scheme

Actor: one hidden layer with 32 `Tanh` units. Critics: one hidden layer with `LeakyReLU` and width 32 (classical) or 512 (Sobolev). Linear layers use Xavier uniform weights without a custom gain and zero biases.

In [ ]:
import torch
import torch.nn as nn


class Actor(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim, d, sigma_min, sigma_max, rho_min, rho_max, pretrained_model=None):
        super().__init__()
        self.d = int(d)
        self.sigma_min, self.sigma_max = list(sigma_min), list(sigma_max)
        if self.d == 2:
            self.rho_min, self.rho_max = list(rho_min), list(rho_max)
        else:
            self.rho_min, self.rho_max = None, None
        self.input_mean, self.input_std = None, None
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.tanh = nn.Tanh()

        if pretrained_model is None:
            for layer in self.net:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    nn.init.zeros_(layer.bias)
        else:
            self.load_state_dict(pretrained_model.state_dict())

    def forward(self, s):
        if (self.input_mean is not None) and (self.input_std is not None):
            s = (s - self.input_mean) / self.input_std
        return self.net(s)

    def squash_vols(self, u):
        sigma_min, sigma_max = u.new_tensor(self.sigma_min).view(1, -1), u.new_tensor(self.sigma_max).view(1, -1)
        shift, scale = 0.5 * (sigma_max + sigma_min), 0.5 * (sigma_max - sigma_min)
        return shift + scale * self.tanh(u)

    def raw_to_pcorr(self, u):
        eps = 1e-2
        return (1.0 - eps) * torch.tanh(u)

    def squash_rho_d2(self, u):
        rho_min, rho_max = u.new_tensor(self.rho_min).view(1, -1), u.new_tensor(self.rho_max).view(1, -1)
        shift, scale = 0.5 * (rho_max + rho_min), 0.5 * (rho_max - rho_min)
        return shift + scale * self.tanh(u)

    @staticmethod
    def _s(x: torch.Tensor) -> torch.Tensor:
        return torch.sqrt((1.0 - x * x).clamp_min(0.0))

    def pcorr_to_corr(self, kappa):
        if self.d == 3:
            rho12, rho13, k23_1 = kappa[:, 0], kappa[:, 1], kappa[:, 2]
            s12, s13 = self._s(rho12), self._s(rho13)
            rho23 = k23_1 * s12 * s13 + rho12 * rho13
            return torch.stack([rho12, rho13, rho23], dim=1)

        if self.d == 4:
            k1, k2, k3, k4, k5, k6 = kappa.unbind(dim=1)
            s1, s2, s3 = self._s(k1), self._s(k2), self._s(k3)
            s4, s5, s6 = self._s(k4), self._s(k5), self._s(k6)
            rho12, rho13, rho14 = k1, k2, k4
            rho23, rho24, rho34 = k1 * k2 + s1 * s2 * k3, k1 * k4 + s1 * s4 * k5, k2 * k4 + s2 * s4 * (k3 * k5 + s3 * s5 * k6)            
            return torch.stack([rho12, rho13, rho14, rho23, rho24, rho34], dim=1)

        if self.d == 5:
            k1, k2, k3, k4, k5, k6, k7, k8, k9, k10 = kappa.unbind(dim=1)
            s1, s2, s3, s4 = self._s(k1), self._s(k2), self._s(k3), self._s(k4)
            s5, s6, s7, s8, s9, s10 = self._s(k5), self._s(k6), self._s(k7), self._s(k8), self._s(k9), self._s(k10)
            rho12, rho13, rho14, rho15 = k1, k2, k3, k4
            rho23, rho24, rho25 = k1 * k2 + s1 * s2 * k5, k1 * k3 + s1 * s3 * k6, k1 * k4 + s1 * s4 * k7
            rho34, rho35 = k2 * k3 + s2 * s3 * (k5 * k6 + s5 * s6 * k8), k2 * k4 + s2 * s4 * (k5 * k7 + s5 * s7 * k9)
            rho45_12 = k10 * (s8 * s9) + k8 * k9
            rho45_1 = rho45_12 * (s6 * s7) + k6 * k7
            rho45 = rho45_1 * (s3 * s4) + k3 * k4
            return torch.stack([rho12, rho13, rho14, rho15, rho23, rho24, rho25, rho34, rho35, rho45], dim=1)

    def pcorr_to_chol(self, kappa):
        """Build the Cholesky factor L of the correlation matrix directly
        from the C-vine partial correlations.  L is lower-triangular with
        positive diagonal, satisfying R = L L^T.  No numerical Cholesky
        factorization is needed."""
        M = kappa.shape[0]
        dev, dt = kappa.device, kappa.dtype

        if self.d == 3:
            k1, k2, k3 = kappa[:, 0], kappa[:, 1], kappa[:, 2]
            s1, s2, s3 = self._s(k1), self._s(k2), self._s(k3)
            L = torch.zeros(M, 3, 3, device=dev, dtype=dt)
            L[:, 0, 0] = 1.0
            L[:, 1, 0] = k1;           L[:, 1, 1] = s1
            L[:, 2, 0] = k2;           L[:, 2, 1] = k3 * s2;      L[:, 2, 2] = s3 * s2
            return L

        if self.d == 4:
            k1, k2, k3, k4, k5, k6 = kappa.unbind(dim=1)
            s1, s2, s3 = self._s(k1), self._s(k2), self._s(k3)
            s4, s5, s6 = self._s(k4), self._s(k5), self._s(k6)
            L = torch.zeros(M, 4, 4, device=dev, dtype=dt)
            L[:, 0, 0] = 1.0
            L[:, 1, 0] = k1;           L[:, 1, 1] = s1
            L[:, 2, 0] = k2;           L[:, 2, 1] = k3 * s2;      L[:, 2, 2] = s3 * s2
            L[:, 3, 0] = k4;           L[:, 3, 1] = k5 * s4;      L[:, 3, 2] = k6 * s5 * s4;     L[:, 3, 3] = s6 * s5 * s4
            return L

        if self.d == 5:
            k1, k2, k3, k4, k5, k6, k7, k8, k9, k10 = kappa.unbind(dim=1)
            s1, s2, s3, s4 = self._s(k1), self._s(k2), self._s(k3), self._s(k4)
            s5, s6, s7, s8, s9, s10 = self._s(k5), self._s(k6), self._s(k7), self._s(k8), self._s(k9), self._s(k10)
            L = torch.zeros(M, 5, 5, device=dev, dtype=dt)
            L[:, 0, 0] = 1.0
            L[:, 1, 0] = k1;           L[:, 1, 1] = s1
            L[:, 2, 0] = k2;           L[:, 2, 1] = k5 * s2;          L[:, 2, 2] = s5 * s2
            L[:, 3, 0] = k3;           L[:, 3, 1] = k6 * s3;          L[:, 3, 2] = k8 * s6 * s3;         L[:, 3, 3] = s8 * s6 * s3
            L[:, 4, 0] = k4;           L[:, 4, 1] = k7 * s4;          L[:, 4, 2] = k9 * s7 * s4;         L[:, 4, 3] = k10 * s9 * s7 * s4;   L[:, 4, 4] = s10 * s9 * s7 * s4
            return L

    def raw_to_chol(self, u):
        """From raw network output to (L, rho).
        Returns the Cholesky factor L and the standard correlations rho."""
        if self.d == 2:
            rho = self.squash_rho_d2(u)
            r12 = rho[:, 0]
            M = rho.shape[0]
            L = torch.zeros(M, 2, 2, device=rho.device, dtype=rho.dtype)
            L[:, 0, 0] = 1.0
            L[:, 1, 0] = r12
            L[:, 1, 1] = self._s(r12)
            return L, rho
        kappa = self.raw_to_pcorr(u)
        rho = self.pcorr_to_corr(kappa)
        L = self.pcorr_to_chol(kappa)
        return L, rho

    def raw_to_rho(self, u):
        if self.d == 2:
            return self.squash_rho_d2(u)
        kappa = self.raw_to_pcorr(u)
        return self.pcorr_to_corr(kappa)

    def mean_bounded(self, s):
        """Returns (action, L) where action = cat([sigma, rho]) and L is
        the Cholesky factor of the correlation matrix."""
        mu_raw = self.forward(s)
        vol_raw, corr_raw = mu_raw[:, : self.d], mu_raw[:, self.d :]
        sigma = self.squash_vols(vol_raw)
        L, rho = self.raw_to_chol(corr_raw)
        action = torch.cat([sigma, rho], dim=1)
        return action, L


In [ ]:
import torch
import torch.nn as nn


class ValueCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, pretrained_model=None):
        super().__init__()
        self.input_mean, self.input_std = None, None
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, 1),
        )

        if pretrained_model is None:
            for layer in self.net:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    nn.init.zeros_(layer.bias)
        else:
            self.load_state_dict(pretrained_model.state_dict())

    def forward(self, s):
        if (self.input_mean is not None) and (self.input_std is not None):
            s = (s - self.input_mean) / self.input_std
        out = self.net(s)
        return out.squeeze(-1)


In [ ]:
import torch


def euler_step(s, sigma, L, Z, dt, r):
    """One step of the log-Euler scheme.
    
    Args:
        s:     current state, shape (M, d)
        sigma: volatility vector, shape (M, d)
        L:     Cholesky factor of the correlation matrix, shape (M, d, d)
        Z:     Brownian increments (already scaled by sqrt(dt)), shape (M, d)
        dt:    time step
        r:     risk-free rate
    """
    W = torch.bmm(L, Z.unsqueeze(-1)).squeeze(-1)
    drift = (r - 0.5 * sigma * sigma) * dt
    return s * torch.exp(sigma * W + drift)


## Classical backward SPG and PPO

The advantage, Gaussian likelihood ratio, clipped PPO objective, annealed exploration and correlation penalty are implemented directly below.

In [ ]:
import math
import torch
import torch.optim as optim



MSE_fun_classical = torch.nn.MSELoss()


def decay_sigmoid_classical(i, I, val_max, val_min, steepness=10.0, mid=0.5):
    if I <= 1:
        return val_min
    t = i / (I - 1)
    device = val_max.device if torch.is_tensor(val_max) else None
    x = torch.as_tensor(steepness * (t - mid), device=device)
    s0 = torch.sigmoid(torch.as_tensor(steepness * (0.0 - mid), device=device))
    s1 = torch.sigmoid(torch.as_tensor(steepness * (1.0 - mid), device=device))
    u = (torch.sigmoid(x) - s0) / (s1 - s0)
    return val_max + (val_min - val_max) * u


def huber_classical(v, d=0.05):
    return torch.where(v <= d, 0.5 * v * v / d, v - 0.5 * d)


def _reward_column_classical(value, n, device):
    """Convert a reward callback output to shape (n, 1) without blocking
    callbacks that use autograd internally."""
    if not torch.is_tensor(value):
        value = torch.as_tensor(value, device=device, dtype=torch.float32)
    else:
        value = value.to(device=device)
    if value.numel() != n:
        raise ValueError(f"Reward callback must return one scalar per state; got {value.numel()} values for {n} states.")
    return value.reshape(n, 1)


def solve_step_k_classical(
    N,
    M,
    d,
    k,
    dt,
    r,
    critic_arch,
    actor_arch,
    I,
    mu,
    critic_list,
    actor_list,
    batch_size,
    lbd_max,
    lbd_min,
    epsilon_clip,
    rho_min,
    rho_max,
    critic_lr_max,
    critic_lr_min,
    actor_lr_max,
    actor_lr_min,
    beta,
    running_reward,
    terminal_reward,
    device,
):
    """Solve the backward actor--critic problem at time t_k.

    Reward contract
    ---------------
    running_reward(states, action, t): one scalar per current state,
        evaluated at physical time t = k*dt and at the physical control
        action = (sigma, rho).
    terminal_reward(states): one scalar per terminal state.

    Both callbacks are required.  The one-step target is

        running_reward * dt + exp(-r dt) * continuation,

    with local antithetic averaging in the continuation term.
    """

    n_rho = d * (d - 1) // 2
    out_dim = d + n_rho
    discount = math.exp(-r * dt)
    t_k = k * dt

    lbd_max = lbd_max.to(device).view(1, out_dim)
    lbd_min = lbd_min.to(device).view(1, out_dim)

    if d >= 3:
        rho_min = rho_min.to(device).view(1, n_rho)
        rho_max = rho_max.to(device).view(1, n_rho)
        corr_w = rho_max - rho_min
    else:
        rho_min, rho_max, corr_w = None, None, None

    critic_arch, actor_arch = critic_arch.copy(), actor_arch.copy()
    critic_arch["pretrained_model"] = None if len(critic_list) == 0 else critic_list[0]
    actor_arch["pretrained_model"] = None if len(actor_list) == 0 else actor_list[0]

    critic = ValueCritic(**critic_arch).to(device)
    actor = Actor(**actor_arch).to(device)
    actor_old = Actor(**actor_arch).to(device)

    use_cuda = device.type == "cuda"
    critic_optimizer = optim.Adam(critic.parameters(), lr=critic_lr_max, fused=use_cuda)
    actor_optimizer = optim.Adam(actor.parameters(), lr=actor_lr_max, fused=use_cuda)

    run_mean, run_var, alpha = None, None, 0.02
    V_scale_sq = None

    for c in critic_list:
        c.eval()
        c.to(device)
        if c.input_mean is not None:
            c.input_mean = c.input_mean.to(device)
        if c.input_std is not None:
            c.input_std = c.input_std.to(device)
        for p in c.parameters():
            p.requires_grad_(False)

    actor_old.eval()
    for p in actor_old.parameters():
        p.requires_grad_(False)

    for m in actor_list:
        m.eval()
        m.to(device)
        if m.input_mean is not None:
            m.input_mean = m.input_mean.to(device)
        if m.input_std is not None:
            m.input_std = m.input_std.to(device)
        for p in m.parameters():
            p.requires_grad_(False)

    def safe_actor_bounded(model):
        def policy(x):
            return model.mean_bounded(x)
        return policy

    def make_stochastic_policy(model, Z_slice, sqrt_lbd):
        def policy(x):
            m = model(x)
            gaussian_raw = m + sqrt_lbd * Z_slice
            vol_raw, corr_raw = gaussian_raw[:, :d], gaussian_raw[:, d:]
            sigma = model.squash_vols(vol_raw)
            L, rho = model.raw_to_chol(corr_raw)
            action = torch.cat([sigma, rho], dim=1)
            return action, L
        return policy

    xi = torch.empty((M, out_dim), device=device)
    dW_base = torch.empty((M, d), device=device)

    for i in range(I):
        # --------------------------------------------------------------
        # Frozen rollout under the old actor.
        # --------------------------------------------------------------
        with torch.no_grad():
            dW_base.normal_().mul_(math.sqrt(dt))
            xi.normal_()

            initial_state = mu(M)
            initial_state_pair = torch.cat([initial_state, initial_state], dim=0)
            xi_pair = torch.cat([xi, xi], dim=0)
            dW_pair = torch.cat([dW_base, -dW_base], dim=0)

            batch_mean = initial_state.mean(0, keepdim=True)
            batch_var = initial_state.var(0, keepdim=True, unbiased=False).clamp_min(1e-12)
            if run_mean is None:
                run_mean, run_var = batch_mean, batch_var
            else:
                run_mean = (1.0 - alpha) * run_mean + alpha * batch_mean
                run_var = (1.0 - alpha) * run_var + alpha * batch_var

            mean_s = run_mean
            std_s = torch.sqrt(run_var + 1e-12)
            actor.input_mean = mean_s
            actor.input_std = std_s
            critic.input_mean = mean_s
            critic.input_std = std_s
            actor_old.input_mean = mean_s
            actor_old.input_std = std_s
            actor_old.load_state_dict(actor.state_dict())

        lbd_i = decay_sigmoid_classical(i, I, lbd_max, lbd_min)
        sqrt_lbd_i = torch.sqrt(lbd_i)
        inv_lbd_i = 1.0 / lbd_i
        stochastic_action = make_stochastic_policy(actor_old, xi_pair, sqrt_lbd_i)

        actor_lr_i = decay_sigmoid_classical(i, I, actor_lr_max, actor_lr_min)
        critic_lr_i = decay_sigmoid_classical(i, I, critic_lr_max, critic_lr_min)
        for p in actor_optimizer.param_groups:
            p["lr"] = actor_lr_i
        for p in critic_optimizer.param_groups:
            p["lr"] = critic_lr_i

        with torch.no_grad():
            S_tj = initial_state
            action_pair, L_k = stochastic_action(initial_state_pair)
            sigma_pair = action_pair[:, :d]
            S_tp1_pair = euler_step(initial_state_pair, sigma_pair, L_k, dW_pair, dt, r)

            # The two antithetic branches have the same state/action at t_k.
            action_at_k = action_pair[:M]

            if len(critic_list) > 0:
                V_tp1 = critic_list[0](S_tp1_pair).view(2, M, 1)
                continuation = V_tp1.mean(dim=0)
            else:
                continuation = None

            m_old_train = actor_old(S_tj)
            gaussian_old_train = m_old_train + sqrt_lbd_i * xi

        # Reward callbacks are deliberately outside torch.no_grad(): an
        # auxiliary reward is allowed to use autograd internally.
        running_val = _reward_column_classical(
            running_reward(S_tj.detach(), action_at_k.detach(), t_k),
            M,
            device,
        ).detach()

        if len(critic_list) > 0:
            critic_target = running_val * dt + discount * continuation.detach()
        else:
            terminal_val = _reward_column_classical(
                terminal_reward(S_tp1_pair.detach()),
                2 * M,
                device,
            ).view(2, M, 1).mean(dim=0).detach()
            critic_target = running_val * dt + discount * terminal_val

        # Freeze the PPO advantage before the current critic update.
        with torch.no_grad():
            advantage_all = critic_target.view(-1) - critic(S_tj)
            advantage_all.sub_(advantage_all.mean()).div_(
                advantage_all.std(unbiased=False) + 1e-12
            )

        if V_scale_sq is None:
            V_scale_sq = (critic_target ** 2).mean().clamp_min(1e-12).item()

        # --------------------------------------------------------------
        # Critic update.
        # --------------------------------------------------------------
        for p in critic.parameters():
            p.requires_grad_(True)
        critic.train()

        num_batch = S_tj.shape[0] // batch_size
        for b in range(num_batch):
            sl = slice(b * batch_size, (b + 1) * batch_size)
            S_b = S_tj[sl]
            critic_target_b = critic_target[sl]

            critic_optimizer.zero_grad(set_to_none=True)
            critic_pred = critic(S_b)
            critic_loss = MSE_fun_classical(critic_pred, critic_target_b.view(-1)) / V_scale_sq
            critic_loss.backward()
            critic_optimizer.step()

        critic.eval()
        for p in critic.parameters():
            p.requires_grad_(False)

        # --------------------------------------------------------------
        # PPO actor update using the already frozen advantage.
        # --------------------------------------------------------------
        actor.train()
        for b in range(num_batch):
            sl = slice(b * batch_size, (b + 1) * batch_size)
            S_b = S_tj[sl]
            gaussian_old_b = gaussian_old_train[sl]
            m_old_b = m_old_train[sl]
            advantage = advantage_all[sl]

            actor_optimizer.zero_grad(set_to_none=True)
            m_new_b = actor(S_b)
            expo = (
                (m_new_b - m_old_b)
                * (gaussian_old_b - 0.5 * (m_new_b + m_old_b))
                * inv_lbd_i
            ).sum(dim=1)
            ratio = torch.exp(torch.clamp(expo, -20.0, 20.0))

            penalty = 0.0
            if d >= 3:
                corr_mean_new = m_new_b[:, d:]
                rho_mean_new = actor.raw_to_rho(corr_mean_new)
                over = torch.clamp(rho_mean_new - rho_max, min=0.0) / (corr_w + 1e-12)
                under = torch.clamp(rho_min - rho_mean_new, min=0.0) / (corr_w + 1e-12)
                penalty = (huber_classical(over) + huber_classical(under)).mean()

            unclipped = ratio * advantage
            clipped = torch.clamp(ratio, 1 - epsilon_clip, 1 + epsilon_clip) * advantage
            actor_loss = -torch.min(unclipped, clipped).mean() + beta * penalty
            actor_loss.backward()
            actor_optimizer.step()

    # ------------------------------------------------------------------
    # Final deterministic-policy critic refit, with the same generic
    # running/terminal reward structure.
    # ------------------------------------------------------------------
    actor.eval()
    deterministic_action = safe_actor_bounded(actor)

    with torch.no_grad():
        dW_base.normal_().mul_(math.sqrt(dt))
        initial_state = mu(M)
        S_tj = initial_state
        initial_state_pair = torch.cat([initial_state, initial_state], dim=0)
        dW_pair = torch.cat([dW_base, -dW_base], dim=0)

        action_pair, L_k = deterministic_action(initial_state_pair)
        sigma_pair = action_pair[:, :d]
        action_at_k = action_pair[:M]
        S_tp1_pair = euler_step(initial_state_pair, sigma_pair, L_k, dW_pair, dt, r)

        if len(critic_list) > 0:
            V_tp1 = critic_list[0](S_tp1_pair).view(2, M, 1)
            continuation = V_tp1.mean(dim=0)
        else:
            continuation = None

    running_val = _reward_column_classical(
        running_reward(S_tj.detach(), action_at_k.detach(), t_k),
        M,
        device,
    ).detach()

    if len(critic_list) > 0:
        critic_target = running_val * dt + discount * continuation.detach()
    else:
        terminal_val = _reward_column_classical(
            terminal_reward(S_tp1_pair.detach()),
            2 * M,
            device,
        ).view(2, M, 1).mean(dim=0).detach()
        critic_target = running_val * dt + discount * terminal_val

    for p in critic.parameters():
        p.requires_grad_(True)
    critic.train()

    num_batch = S_tj.shape[0] // batch_size
    for b in range(num_batch):
        sl = slice(b * batch_size, (b + 1) * batch_size)
        S_b = S_tj[sl]
        critic_target_b = critic_target[sl]

        critic_optimizer.zero_grad(set_to_none=True)
        critic_pred = critic(S_b)
        critic_loss = MSE_fun_classical(critic_pred, critic_target_b.view(-1)) / V_scale_sq
        critic_loss.backward()
        critic_optimizer.step()

    critic.eval()
    for p in critic.parameters():
        p.requires_grad_(False)

    return critic, actor


In [ ]:
import math
import torch



def train_all_classical(
    T,
    N,
    M,
    S_0,
    r,
    critic_arch,
    actor_arch,
    batch_size,
    I_outer,
    I_inner,
    sigma_min,
    sigma_max,
    rho_min,
    rho_max,
    lbd_max,
    lbd_min,
    epsilon_clip,
    critic_lr_max_1,
    critic_lr_max_2,
    critic_lr_min_1,
    critic_lr_min_2,
    actor_lr_max_1,
    actor_lr_max_2,
    actor_lr_min_1,
    actor_lr_min_2,
    beta,
    running_reward,
    terminal_reward,
):
    

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    S_0 = S_0.to(device).float()
    d = int(S_0.shape[0])   
    dt = T / N   
    sigma_max = torch.as_tensor(sigma_max, device=device, dtype=torch.float32).view(d)
    sigma_min = torch.as_tensor(sigma_min, device=device, dtype=torch.float32).view(d)
    
    def _sample_regression_states(M_, t_):
        sigma = (sigma_min + (sigma_max - sigma_min) * torch.rand(M_, d, device=device))
        drift = (r - 0.5 * sigma * sigma) * t_
        effective_sqrt = math.sqrt(t_ + 0.1**2)
        diff = (sigma * effective_sqrt * torch.randn(M_, d, device=device))
        return torch.exp(torch.log(S_0).view(1, d) + drift + diff)
    
    critic_list, actor_list = [], []

    for k in reversed(range(N)):
        t_k = k * dt

        def mu(M_, t_k_=t_k):
            return _sample_regression_states(M_, t_k_)

        if k == N - 1:
            I_step = I_outer
            critic_lr_max_step = critic_lr_max_1
            actor_lr_max_step = actor_lr_max_1
            critic_lr_min_step = critic_lr_min_1
            actor_lr_min_step = actor_lr_min_1
        else:
            I_step = I_inner
            critic_lr_max_step = critic_lr_max_2
            actor_lr_max_step = actor_lr_max_2
            critic_lr_min_step = critic_lr_min_2
            actor_lr_min_step = actor_lr_min_2

        critic_k, actor_k = solve_step_k_classical(
            N,
            M,
            d,
            k,
            dt,
            r,
            critic_arch,
            actor_arch,
            I_step,
            mu,
            critic_list,
            actor_list,
            batch_size,
            lbd_max,
            lbd_min,
            epsilon_clip,
            rho_min,
            rho_max,
            critic_lr_max_step,
            critic_lr_min_step,
            actor_lr_max_step,
            actor_lr_min_step,
            beta,
            running_reward,
            terminal_reward,
            device,
        )

        actor_k.eval()
        critic_list = [critic_k] + critic_list
        actor_list = [actor_k] + actor_list

    return critic_list, actor_list


## Sobolev backward SPG and PPO

This branch trains independent networks. At the final backward step, only the spatial-gradient labels use centered finite differences (step 0.1 in asset-price units) with controls and Gaussian draws held fixed. The Bellman value label uses the exact terminal payoff. At all earlier steps, spatial-gradient labels use automatic differentiation as before. The classical training branch is unchanged.


In [ ]:
import math
import torch
import torch.optim as optim



MSE_fun_sobolev = torch.nn.MSELoss()


def decay_sigmoid_sobolev(i, I, val_max, val_min, steepness=10.0, mid=0.5):
    if I <= 1:
        return val_min
    t = i / (I - 1)
    device = val_max.device if torch.is_tensor(val_max) else None
    x = torch.as_tensor(steepness * (t - mid), device=device)
    s0 = torch.sigmoid(torch.as_tensor(steepness * (0.0 - mid), device=device))
    s1 = torch.sigmoid(torch.as_tensor(steepness * (1.0 - mid), device=device))
    u = (torch.sigmoid(x) - s0) / (s1 - s0)
    return val_max + (val_min - val_max) * u


def huber_sobolev(v, d=0.05):
    return torch.where(v <= d, 0.5 * v * v / d, v - 0.5 * d)


def _set_requires_grad_sobolev(model, flag):
    for p in model.parameters():
        p.requires_grad_(flag)


def _reward_column_sobolev(value, n, device, dtype):
    """Return one reward per state, preserving an existing autograd graph."""
    if not torch.is_tensor(value):
        value = torch.as_tensor(value, device=device, dtype=dtype)
    else:
        value = value.to(device=device, dtype=dtype)
    if value.numel() != n:
        raise ValueError(
            f"Reward callback must return one scalar per state; "
            f"got {value.numel()} values for {n} states."
        )
    return value.reshape(n)


def _physical_action_sobolev(actor, state, xi, sqrt_lambda, d, stochastic):
    """Physical action and correlation Cholesky factor at ``state``."""
    m = actor(state)
    raw = m + sqrt_lambda * xi if stochastic else m
    vol_raw, corr_raw = raw[:, :d], raw[:, d:]
    sigma = actor.squash_vols(vol_raw)
    L, rho = actor.raw_to_chol(corr_raw)
    action = torch.cat([sigma, rho], dim=1)
    return m, action, L


def _bellman_value_given_action_sobolev(
    state,
    action,
    L,
    dW,
    d,
    discount,
    r,
    dt,
    t_k,
    continuation_critic,
    running_reward,
    terminal_reward,
):
    """One-step Bellman map for a prescribed physical action."""
    sigma = action[:, :d]

    next_plus = euler_step(state, sigma, L, dW, dt, r)
    next_minus = euler_step(state, sigma, L, -dW, dt, r)

    reward = _reward_column_sobolev(
        running_reward(state, action, t_k),
        state.shape[0],
        state.device,
        state.dtype,
    )

    if continuation_critic is None:
        cont_plus = _reward_column_sobolev(
            terminal_reward(next_plus),
            state.shape[0],
            state.device,
            state.dtype,
        )
        cont_minus = _reward_column_sobolev(
            terminal_reward(next_minus),
            state.shape[0],
            state.device,
            state.dtype,
        )
    else:
        cont_plus = continuation_critic(next_plus)
        cont_minus = continuation_critic(next_minus)

    target = reward * dt + discount * 0.5 * (cont_plus + cont_minus)
    return target, next_plus


def _bellman_value_sobolev(
    state,
    actor,
    xi,
    sqrt_lambda,
    dW,
    d,
    discount,
    r,
    dt,
    t_k,
    continuation_critic,
    running_reward,
    terminal_reward,
    stochastic,
):
    """Full local-antithetic one-step Bellman map at fixed random inputs."""
    m, action, L = _physical_action_sobolev(
        actor, state, xi, sqrt_lambda, d, stochastic
    )
    target, next_plus = _bellman_value_given_action_sobolev(
        state,
        action,
        L,
        dW,
        d,
        discount,
        r,
        dt,
        t_k,
        continuation_critic,
        running_reward,
        terminal_reward,
    )
    return m, target, next_plus


def _interior_target_aad_sobolev(
    state,
    actor,
    xi,
    sqrt_lambda,
    dW,
    d,
    discount,
    r,
    dt,
    t_k,
    continuation_critic,
    running_reward,
    stochastic,
):
    """Interior Bellman value and spatial-gradient label at fixed action.

    The actor is evaluated once at the base state.  The resulting physical
    action is detached and held fixed when differentiating the one-step
    objective with respect to the state, in accordance with the envelope
    argument used in the paper.
    """
    state0 = state.detach()
    with torch.no_grad():
        _, action, L = _physical_action_sobolev(
            actor, state0, xi, sqrt_lambda, d, stochastic
        )
        action = action.detach()
        L = L.detach()

    state_req = state0.clone().requires_grad_(True)
    target, next_plus = _bellman_value_given_action_sobolev(
        state_req,
        action,
        L,
        dW,
        d,
        discount,
        r,
        dt,
        t_k,
        continuation_critic,
        running_reward,
        None,
    )
    dtarget = torch.autograd.grad(
        target.sum(), state_req, create_graph=False
    )[0]
    return (
        target.detach().view(-1, 1),
        dtarget.detach(),
        next_plus.detach(),
    )


def _terminal_target_fd_sobolev(
    state,
    actor,
    xi,
    sqrt_lambda,
    dW,
    d,
    discount,
    r,
    dt,
    t_k,
    running_reward,
    terminal_reward,
    stochastic,
):
    """Final Bellman value with centered-FD labels at fixed action.

    The exact nonsmooth terminal payoff remains in the value target.  Only the
    Sobolev labels are finite-differenced.  The same dW/xi and the same physical
    action, evaluated once at the base state, are reused for all perturbations.
    """
    fd_delta = 0.1
    if not math.isfinite(fd_delta) or fd_delta <= 0:
        raise ValueError("Terminal finite-difference step must be positive and finite")
    state0 = state.detach()
    with torch.no_grad():
        _, action, L = _physical_action_sobolev(
            actor, state0, xi, sqrt_lambda, d, stochastic
        )
        action = action.detach()
        L = L.detach()

        target0, next_plus = _bellman_value_given_action_sobolev(
            state0,
            action,
            L,
            dW,
            d,
            discount,
            r,
            dt,
            t_k,
            None,
            running_reward,
            terminal_reward,
        )

        grad_cols = []
        for j in range(d):
            shift = torch.zeros_like(state0)
            shift[:, j] = fd_delta
            state_plus = (state0 + shift).clamp_min(1e-8)
            state_minus = (state0 - shift).clamp_min(1e-8)

            target_plus, _ = _bellman_value_given_action_sobolev(
                state_plus,
                action,
                L,
                dW,
                d,
                discount,
                r,
                dt,
                t_k,
                None,
                running_reward,
                terminal_reward,
            )
            target_minus, _ = _bellman_value_given_action_sobolev(
                state_minus,
                action,
                L,
                dW,
                d,
                discount,
                r,
                dt,
                t_k,
                None,
                running_reward,
                terminal_reward,
            )
            grad_cols.append(
                (target_plus - target_minus) / (2.0 * fd_delta)
            )

    dtarget = torch.stack(grad_cols, dim=1)
    return target0.view(-1, 1), dtarget, next_plus


def _critic_update_sobolev(
    critic,
    critic_optimizer,
    states,
    V_target,
    dV_target,
    batch_size,
    V_scale_sq,
    dV_scale_sq,
    eta_V,
    eta_dV,
):
    _set_requires_grad_sobolev(critic, True)
    critic.train()

    num_batch = states.shape[0] // batch_size
    for b in range(num_batch):
        sl = slice(b * batch_size, (b + 1) * batch_size)
        S_b = states[sl].detach().clone().requires_grad_(True)
        V_b = V_target[sl].view(-1)
        dV_b = dV_target[sl]

        critic_optimizer.zero_grad(set_to_none=True)
        V_pred = critic(S_b)
        dV_pred = torch.autograd.grad(
            V_pred.sum(), S_b, create_graph=True
        )[0]

        loss_V = MSE_fun_sobolev(V_pred, V_b)
        loss_dV = MSE_fun_sobolev(dV_pred, dV_b)
        loss = (
            eta_V * loss_V / V_scale_sq
            + eta_dV * loss_dV / dV_scale_sq
        )
        loss.backward()
        critic_optimizer.step()

    critic.eval()
    _set_requires_grad_sobolev(critic, False)


def solve_step_k_sobolev(
    N,
    M,
    d,
    k,
    dt,
    r,
    critic_arch,
    actor_arch,
    I,
    mu,
    critic_list,
    actor_list,
    batch_size,
    lbd_max,
    lbd_min,
    epsilon_clip,
    rho_min,
    rho_max,
    critic_lr_max,
    critic_lr_min,
    actor_lr_max,
    actor_lr_min,
    eta_V,
    eta_dV,
    beta,
    running_reward,
    terminal_reward,
    device,
):
    """Backward first-order Sobolev SPG step for 2 <= d <= 5."""
    n_rho = d * (d - 1) // 2
    out_dim = d + n_rho
    discount = math.exp(-r * dt)
    t_k = k * dt
    terminal = len(critic_list) == 0

    lbd_max = torch.as_tensor(lbd_max, device=device, dtype=torch.float32).view(1, out_dim)
    lbd_min = torch.as_tensor(lbd_min, device=device, dtype=torch.float32).view(1, out_dim)

    if d >= 3:
        rho_min = torch.as_tensor(rho_min, device=device, dtype=torch.float32).view(1, n_rho)
        rho_max = torch.as_tensor(rho_max, device=device, dtype=torch.float32).view(1, n_rho)
        corr_w = rho_max - rho_min
    else:
        rho_min, rho_max, corr_w = None, None, None

    critic_arch, actor_arch = critic_arch.copy(), actor_arch.copy()
    critic_arch["pretrained_model"] = None if terminal else critic_list[0]
    actor_arch["pretrained_model"] = None if len(actor_list) == 0 else actor_list[0]

    critic = ValueCritic(**critic_arch).to(device)
    actor = Actor(**actor_arch).to(device)
    actor_old = Actor(**actor_arch).to(device)

    use_cuda = device.type == "cuda"
    critic_optimizer = optim.Adam(
        critic.parameters(), lr=float(critic_lr_max), fused=use_cuda
    )
    actor_optimizer = optim.Adam(
        actor.parameters(), lr=float(actor_lr_max), fused=use_cuda
    )

    run_mean, run_var, alpha = None, None, 0.02
    V_scale_sq, dV_scale_sq = None, None

    continuation_critic = None if terminal else critic_list[0]
    if continuation_critic is not None:
        continuation_critic.eval().to(device)
        if continuation_critic.input_mean is not None:
            continuation_critic.input_mean = continuation_critic.input_mean.to(device)
        if continuation_critic.input_std is not None:
            continuation_critic.input_std = continuation_critic.input_std.to(device)
        _set_requires_grad_sobolev(continuation_critic, False)

    actor_old.eval()
    _set_requires_grad_sobolev(actor_old, False)

    for m in actor_list:
        m.eval().to(device)
        if m.input_mean is not None:
            m.input_mean = m.input_mean.to(device)
        if m.input_std is not None:
            m.input_std = m.input_std.to(device)
        _set_requires_grad_sobolev(m, False)

    xi = torch.empty((M, out_dim), device=device)
    dW = torch.empty((M, d), device=device)

    for i in range(I):
        with torch.no_grad():
            xi.normal_()
            dW.normal_().mul_(math.sqrt(dt))
            S_tj = mu(M)

            batch_mean = S_tj.mean(0, keepdim=True)
            batch_var = S_tj.var(0, keepdim=True, unbiased=False).clamp_min(1e-12)
            if run_mean is None:
                run_mean, run_var = batch_mean, batch_var
            else:
                run_mean = (1.0 - alpha) * run_mean + alpha * batch_mean
                run_var = (1.0 - alpha) * run_var + alpha * batch_var

            mean_s = run_mean
            std_s = torch.sqrt(run_var + 1e-12)
            actor.input_mean = mean_s
            actor.input_std = std_s
            critic.input_mean = mean_s
            critic.input_std = std_s
            actor_old.input_mean = mean_s
            actor_old.input_std = std_s
            actor_old.load_state_dict(actor.state_dict())

        lbd_i = decay_sigmoid_sobolev(i, I, lbd_max, lbd_min)
        sqrt_lbd_i = torch.sqrt(lbd_i)
        inv_lbd_i = 1.0 / lbd_i

        actor_lr_i = decay_sigmoid_sobolev(i, I, actor_lr_max, actor_lr_min)
        critic_lr_i = decay_sigmoid_sobolev(i, I, critic_lr_max, critic_lr_min)
        for p in actor_optimizer.param_groups:
            p["lr"] = actor_lr_i
        for p in critic_optimizer.param_groups:
            p["lr"] = critic_lr_i

        if terminal:
            V_target, dV_target, _ = _terminal_target_fd_sobolev(
                S_tj,
                actor_old,
                xi,
                sqrt_lbd_i,
                dW,
                d,
                discount,
                r,
                dt,
                t_k,
                running_reward,
                terminal_reward,
                stochastic=True,
            )
        else:
            V_target, dV_target, _ = _interior_target_aad_sobolev(
                S_tj,
                actor_old,
                xi,
                sqrt_lbd_i,
                dW,
                d,
                discount,
                r,
                dt,
                t_k,
                continuation_critic,
                running_reward,
                stochastic=True,
            )

        if V_scale_sq is None:
            V_scale_sq = V_target.square().mean().clamp_min(1e-12).item()
            dV_scale_sq = dV_target.square().mean().clamp_min(1e-12).item()

        # Freeze the PPO advantage before the current critic update.
        with torch.no_grad():
            advantage_all = V_target.view(-1) - critic(S_tj)
            advantage_all.sub_(advantage_all.mean()).div_(
                advantage_all.std(unbiased=False) + 1e-12
            )
            m_old_train = actor_old(S_tj)
            gaussian_old_train = m_old_train + sqrt_lbd_i * xi

        _critic_update_sobolev(
            critic,
            critic_optimizer,
            S_tj,
            V_target,
            dV_target,
            batch_size,
            V_scale_sq,
            dV_scale_sq,
            eta_V,
            eta_dV,
        )

        actor.train()
        num_batch = S_tj.shape[0] // batch_size
        for b in range(num_batch):
            sl = slice(b * batch_size, (b + 1) * batch_size)
            S_b = S_tj[sl]
            gaussian_old_b = gaussian_old_train[sl]
            m_old_b = m_old_train[sl]
            advantage = advantage_all[sl]

            actor_optimizer.zero_grad(set_to_none=True)
            m_new_b = actor(S_b)
            expo = (
                (m_new_b - m_old_b)
                * (gaussian_old_b - 0.5 * (m_new_b + m_old_b))
                * inv_lbd_i
            ).sum(dim=1)
            ratio = torch.exp(torch.clamp(expo, -20.0, 20.0))

            penalty = 0.0
            if d >= 3:
                corr_mean_new = m_new_b[:, d:]
                rho_mean_new = actor.raw_to_rho(corr_mean_new)
                over = torch.clamp(rho_mean_new - rho_max, min=0.0) / (corr_w + 1e-12)
                under = torch.clamp(rho_min - rho_mean_new, min=0.0) / (corr_w + 1e-12)
                penalty = (huber_sobolev(over) + huber_sobolev(under)).mean()

            unclipped = ratio * advantage
            clipped = torch.clamp(ratio, 1 - epsilon_clip, 1 + epsilon_clip) * advantage
            actor_loss = -torch.min(unclipped, clipped).mean() + beta * penalty
            actor_loss.backward()
            actor_optimizer.step()

    # Final deterministic-policy Sobolev critic refit.
    actor.eval()
    _set_requires_grad_sobolev(actor, False)

    with torch.no_grad():
        dW.normal_().mul_(math.sqrt(dt))
        S_tj = mu(M)

    if terminal:
        V_target, dV_target, _ = _terminal_target_fd_sobolev(
            S_tj,
            actor,
            None,
            None,
            dW,
            d,
            discount,
            r,
            dt,
            t_k,
            running_reward,
            terminal_reward,
            stochastic=False,
        )
    else:
        V_target, dV_target, _ = _interior_target_aad_sobolev(
            S_tj,
            actor,
            None,
            None,
            dW,
            d,
            discount,
            r,
            dt,
            t_k,
            continuation_critic,
            running_reward,
            stochastic=False,
        )

    _critic_update_sobolev(
        critic,
        critic_optimizer,
        S_tj,
        V_target,
        dV_target,
        batch_size,
        V_scale_sq,
        dV_scale_sq,
        eta_V,
        eta_dV,
    )

    return critic, actor


In [ ]:
import math
import torch



def train_all_sobolev(
    T,
    N,
    M,
    S_0,
    r,
    critic_arch,
    actor_arch,
    batch_size,
    I_outer,
    I_inner,
    sigma_min,
    sigma_max,
    rho_min,
    rho_max,
    lbd_max,
    lbd_min,
    epsilon_clip,
    critic_lr_max_1,
    critic_lr_max_2,
    critic_lr_min_1,
    critic_lr_min_2,
    actor_lr_max_1,
    actor_lr_max_2,
    actor_lr_min_1,
    actor_lr_min_2,
    eta_V,
    eta_dV,
    beta,
    running_reward,
    terminal_reward,
):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    S_0 = S_0.to(device).float()
    d = int(S_0.shape[0])   
    dt = T / N   
    sigma_max = torch.as_tensor(sigma_max, device=device, dtype=torch.float32).view(d)
    sigma_min = torch.as_tensor(sigma_min, device=device, dtype=torch.float32).view(d)
    
    def _sample_regression_states(M_, t_):
        sigma = (sigma_min + (sigma_max - sigma_min) * torch.rand(M_, d, device=device))
        drift = (r - 0.5 * sigma * sigma) * t_
        effective_sqrt = math.sqrt(t_ + 0.1**2)
        diff = (sigma * effective_sqrt * torch.randn(M_, d, device=device))
        return torch.exp(torch.log(S_0).view(1, d) + drift + diff)
    
    critic_list, actor_list = [], []

    for k in reversed(range(N)):
        t_k = k * dt

        def mu(M_, t_k_=t_k):
            return _sample_regression_states(M_, t_k_)

        if k == N - 1:
            I_step = I_outer
            critic_lr_max_step = critic_lr_max_1
            actor_lr_max_step = actor_lr_max_1
            critic_lr_min_step = critic_lr_min_1
            actor_lr_min_step = actor_lr_min_1
        else:
            I_step = I_inner
            critic_lr_max_step = critic_lr_max_2
            actor_lr_max_step = actor_lr_max_2
            critic_lr_min_step = critic_lr_min_2
            actor_lr_min_step = actor_lr_min_2

        critic_k, actor_k = solve_step_k_sobolev(
            N,
            M,
            d,
            k,
            dt,
            r,
            critic_arch,
            actor_arch,
            I_step,
            mu,
            critic_list,
            actor_list,
            batch_size,
            lbd_max,
            lbd_min,
            epsilon_clip,
            rho_min,
            rho_max,
            critic_lr_max_step,
            critic_lr_min_step,
            actor_lr_max_step,
            actor_lr_min_step,
            eta_V,
            eta_dV,
            beta,
            running_reward,
            terminal_reward,
            device,
        )

        actor_k.eval()
        critic_list = [critic_k] + critic_list
        actor_list = [actor_k] + actor_list

    return critic_list, actor_list


## Independent primal evaluation

In [ ]:
import math
import torch


def _reward_column_primal(value, n, device, dtype):
    if not torch.is_tensor(value):
        value = torch.as_tensor(value, device=device, dtype=dtype)
    else:
        value = value.to(device=device, dtype=dtype)
    if value.numel() != n:
        raise ValueError(
            f"Reward callback must return one scalar per state; got "
            f"{value.numel()} values for {n} states."
        )
    return value.reshape(n)


def price_from_actors(
    optimal_controls,
    S0,
    r,
    T,
    N,
    M_eval,
    running_reward,
    terminal_reward,
    rho_min=None,
    rho_max=None,
):
    """Independent MC evaluation of a backward actor sequence, 2 <= d <= 5.

    The evaluated objective is

        E[ sum_k exp(-r t_k) f(t_k, X_k, a_k) dt
           + exp(-r T) h(X_N) ].

    ``running_reward(states, action, t)`` and ``terminal_reward(states)`` are
    required.  Reward callbacks are evaluated outside inference mode so they
    may use autograd internally (for example, candidate residuals/Hessians).
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    S0 = S0.to(device).float()
    d = int(S0.shape[0])
    if not (2 <= d <= 5):
        raise ValueError(f"This evaluator supports 2 <= d <= 5; got d={d}.")
    if len(optimal_controls) != N:
        raise ValueError(
            f"Expected exactly N={N} actors; got {len(optimal_controls)}."
        )

    dt = T / N
    sqrt_dt = math.sqrt(dt)
    n_rho = d * (d - 1) // 2

    actors = []
    for actor in optimal_controls:
        actor = actor.to(device).eval()
        if actor.input_mean is not None:
            actor.input_mean = actor.input_mean.to(device)
        if actor.input_std is not None:
            actor.input_std = actor.input_std.to(device)
        actors.append(actor)

    S = S0.view(1, d).expand(M_eval, d).clone()
    total_reward = torch.zeros(M_eval, device=device, dtype=torch.float32)

    for k in range(N):
        t_k = k * dt

        with torch.no_grad():
            action_k, L_k = actors[k].mean_bounded(S)

        running_val = _reward_column_primal(
            running_reward(S.detach(), action_k.detach(), t_k),
            M_eval,
            device,
            S.dtype,
        ).detach()
        total_reward.add_(math.exp(-r * t_k) * running_val * dt)

        with torch.no_grad():
            sigma_k = action_k[:, :d]
            rho_k = action_k[:, d:]
            Z = torch.randn(M_eval, d, device=device, dtype=S.dtype) * sqrt_dt
            W = torch.bmm(L_k, Z.unsqueeze(-1)).squeeze(-1)
            drift = (r - 0.5 * sigma_k.square()) * dt
            S.mul_(torch.exp(sigma_k * W + drift))

    terminal_val = _reward_column_primal(
        terminal_reward(S.detach()),
        M_eval,
        device,
        S.dtype,
    ).detach()
    total_reward.add_(math.exp(-r * T) * terminal_val)

    estimate = total_reward.mean()
    se = torch.sqrt(total_reward.var(unbiased=True) / M_eval)
    z = 1.96
    ci_low = estimate - z * se
    ci_high = estimate + z * se

    return estimate, ci_low, ci_high


## Approximate pathwise dual evaluation

Uses the C-vine control map, actor initialization, a discrete coordinate grid, a penalized objective for correlation violations in dimensions 3 to 5, and cyclic coordinate sweeps.

In [ ]:
import math
import torch


def _u_grid(grid_size, device):
    eps = 1e-6
    return torch.linspace(-1.0 + eps, 1.0 - eps, grid_size, device=device)
    

def _u_hnet(h, device, dtype):
    eps = 1e-6
    length = 2.0 - 2.0 * eps
    G = max(2, math.ceil(length / h) + 1)
    return torch.linspace(-1.0 + eps, 1.0 - eps, G, device=device, dtype=dtype)


def _s(x):
    return torch.sqrt((1.0 - x * x).clamp_min(0.0))


def _sigma_from_u(u_vol, sigma_min, sigma_max):
    mid = 0.5 * (sigma_max + sigma_min)
    half = 0.5 * (sigma_max - sigma_min)
    return mid + half * u_vol


def _controls_from_u(u, d, sigma_min, sigma_max, rho_min, rho_max):
    u_vol, u_corr = u[..., :d], u[..., d:]
    sigma = _sigma_from_u(u_vol, sigma_min, sigma_max)
    shape = u.shape[:-1]
    dev, dtp = u.device, u.dtype

    if d == 2:
        rho_mid = 0.5 * (rho_max + rho_min)
        rho_half = 0.5 * (rho_max - rho_min)
        rho = rho_mid + rho_half * u_corr
        r12 = rho[..., 0]
        L = torch.zeros(*shape, 2, 2, device=dev, dtype=dtp)
        L[..., 0, 0] = 1.0
        L[..., 1, 0] = r12
        L[..., 1, 1] = _s(r12)
        return sigma, L, rho

    eps_pc = 1e-2
    kappa = (1.0 - eps_pc) * u_corr

    if d == 3:
        k1, k2, k3 = kappa[..., 0], kappa[..., 1], kappa[..., 2]
        s1, s2, s3 = _s(k1), _s(k2), _s(k3)
        L = torch.zeros(*shape, 3, 3, device=dev, dtype=dtp)
        L[..., 0, 0] = 1.0
        L[..., 1, 0] = k1
        L[..., 1, 1] = s1
        L[..., 2, 0] = k2
        L[..., 2, 1] = k3 * s2
        L[..., 2, 2] = s3 * s2
        rho_std = torch.stack([k1, k2, k3 * s1 * s2 + k1 * k2], dim=-1)
        return sigma, L, rho_std

    if d == 4:
        k1, k2, k3, k4, k5, k6 = [kappa[..., i] for i in range(6)]
        s1, s2, s3 = _s(k1), _s(k2), _s(k3)
        s4, s5, s6 = _s(k4), _s(k5), _s(k6)
        L = torch.zeros(*shape, 4, 4, device=dev, dtype=dtp)
        L[..., 0, 0] = 1.0
        L[..., 1, 0] = k1
        L[..., 1, 1] = s1
        L[..., 2, 0] = k2
        L[..., 2, 1] = k3 * s2
        L[..., 2, 2] = s3 * s2
        L[..., 3, 0] = k4
        L[..., 3, 1] = k5 * s4
        L[..., 3, 2] = k6 * s5 * s4
        L[..., 3, 3] = s6 * s5 * s4
        rho_std = torch.stack([
            k1, k2, k4,
            k1 * k2 + s1 * s2 * k3,
            k1 * k4 + s1 * s4 * k5,
            k2 * k4 + s2 * s4 * (k3 * k5 + s3 * s5 * k6),
        ], dim=-1)
        return sigma, L, rho_std

    if d == 5:
        k1, k2, k3, k4, k5, k6, k7, k8, k9, k10 = [kappa[..., i] for i in range(10)]
        s1, s2, s3, s4 = _s(k1), _s(k2), _s(k3), _s(k4)
        s5, s6, s7, s8, s9, s10 = _s(k5), _s(k6), _s(k7), _s(k8), _s(k9), _s(k10)
        L = torch.zeros(*shape, 5, 5, device=dev, dtype=dtp)
        L[..., 0, 0] = 1.0
        L[..., 1, 0] = k1
        L[..., 1, 1] = s1
        L[..., 2, 0] = k2
        L[..., 2, 1] = k5 * s2
        L[..., 2, 2] = s5 * s2
        L[..., 3, 0] = k3
        L[..., 3, 1] = k6 * s3
        L[..., 3, 2] = k8 * s6 * s3
        L[..., 3, 3] = s8 * s6 * s3
        L[..., 4, 0] = k4
        L[..., 4, 1] = k7 * s4
        L[..., 4, 2] = k9 * s7 * s4
        L[..., 4, 3] = k10 * s9 * s7 * s4
        L[..., 4, 4] = s10 * s9 * s7 * s4
        rho45_12 = k10 * s8 * s9 + k8 * k9
        rho45_1 = rho45_12 * s6 * s7 + k6 * k7
        rho45 = rho45_1 * s3 * s4 + k3 * k4
        rho_std = torch.stack([
            k1, k2, k3, k4,
            k1 * k2 + s1 * s2 * k5,
            k1 * k3 + s1 * s3 * k6,
            k1 * k4 + s1 * s4 * k7,
            k2 * k3 + s2 * s3 * (k5 * k6 + s5 * s6 * k8),
            k2 * k4 + s2 * s4 * (k5 * k7 + s5 * s7 * k9),
            rho45,
        ], dim=-1)
        return sigma, L, rho_std

    raise ValueError(f"Unsupported dimension d={d}")


def _log_euler_step(S, sigma, L, xi_n, dt, r):
    sqrt_dt = math.sqrt(dt)
    Z = sqrt_dt * xi_n
    W = torch.einsum("...ij,...j->...i", L, Z)
    drift = (r - 0.5 * sigma * sigma) * dt
    return S * torch.exp(sigma * W + drift)


def _huber(v, delta=0.05):
    return torch.where(v <= delta, 0.5 * v * v / delta, v - 0.5 * delta)




def _penalty_step(rho_std, d, rho_min, rho_max, beta):
    if d < 3 or beta <= 0.0 or rho_min is None:
        return 0.0
    corr_w = (rho_max - rho_min).clamp_min(1e-12)
    over = torch.clamp(rho_std - rho_max, min=0.0) / corr_w
    under = torch.clamp(rho_min - rho_std, min=0.0) / corr_w
    return beta * 2.0 / (d * (d - 1)) * (_huber(over) + _huber(under)).sum(dim=-1)


def _discounted_increment(S_n, S_np1, n, dt, r):
    disc_n = math.exp(-r * n * dt)
    disc_np1 = math.exp(-r * (n + 1) * dt)
    return disc_np1 * S_np1 - disc_n * S_n


def _mart_correction_exact(phi_n, S_n, S_np1, n, dt, r):
    dS_disc = _discounted_increment(S_n, S_np1, n, dt, r)
    return (phi_n * dS_disc).sum(dim=-1)



def _build_cache(u_opt, xi, phi, x0, d, sigma_min, sigma_max,
                 rho_min, rho_max, N, T, r, beta):
    M = u_opt.shape[0]
    dt = T / N
    dev, dtp = u_opt.device, u_opt.dtype

    states = torch.zeros(M, N + 1, d, device=dev, dtype=dtp)
    states[:, 0, :] = x0.unsqueeze(0)
    mart_cum = torch.zeros(M, N + 1, device=dev, dtype=dtp)
    pen_cum = torch.zeros(M, N + 1, device=dev, dtype=dtp)

    for n in range(N):
        S_n = states[:, n, :]
        u_n = u_opt[:, n, :]
        sigma_n, L_n, rho_std_n = _controls_from_u(
            u_n, d, sigma_min, sigma_max, rho_min, rho_max
        )
        xi_n = xi[:, n, :]
        phi_n = phi(n, S_n)

        S_np1 = _log_euler_step(S_n, sigma_n, L_n, xi_n, dt, r)
        corr = _mart_correction_exact(phi_n, S_n, S_np1, n, dt, r)
        pen = _penalty_step(rho_std_n, d, rho_min, rho_max, beta)

        mart_cum[:, n + 1] = mart_cum[:, n] + corr
        disc_n = math.exp(-r * n * dt)
        pen_cum[:, n + 1] = pen_cum[:, n] + (disc_n * dt * pen if torch.is_tensor(pen) else 0.0)
        states[:, n + 1, :] = S_np1

    return states, mart_cum, pen_cum


def _update_cache_from(start_n, states, mart_cum, pen_cum, u_opt, xi, phi, d, sigma_min, sigma_max, rho_min, rho_max, N, T, r, beta):
    dt = T / N

    for n in range(start_n, N):
        S_n = states[:, n, :]
        u_n = u_opt[:, n, :]
        sigma_n, L_n, rho_std_n = _controls_from_u(
            u_n, d, sigma_min, sigma_max, rho_min, rho_max
        )
        xi_n = xi[:, n, :]
        phi_n = phi(n, S_n)

        S_np1 = _log_euler_step(S_n, sigma_n, L_n, xi_n, dt, r)
        corr = _mart_correction_exact(phi_n, S_n, S_np1, n, dt, r)
        pen = _penalty_step(rho_std_n, d, rho_min, rho_max, beta)

        mart_cum[:, n + 1] = mart_cum[:, n] + corr
        disc_n = math.exp(-r * n * dt)
        pen_cum[:, n + 1] = pen_cum[:, n] + (disc_n * dt * pen if torch.is_tensor(pen) else 0.0)
        states[:, n + 1, :] = S_np1




def _forward_partial(start_n, S_start, u_at_start, u_opt, xi, phi, d, sigma_min, sigma_max, rho_min, rho_max, N, T, r, beta):
    G, M, _ = S_start.shape
    dt = T / N
    dev, dtp = S_start.device, S_start.dtype

    S = S_start.clone()
    mart = torch.zeros(G, M, device=dev, dtype=dtp)
    pen_sum = torch.zeros(G, M, device=dev, dtype=dtp)

    for idx in range(N - start_n):
        n = start_n + idx
        if idx == 0:
            u_n = u_at_start
        else:
            u_n = u_opt[:, n, :].unsqueeze(0).expand(G, -1, -1)

        sigma_n, L_n, rho_std_n = _controls_from_u(u_n, d, sigma_min, sigma_max, rho_min, rho_max)
        xi_n = xi[:, n, :].unsqueeze(0).expand(G, -1, -1)

        phi_n = phi(n, S.reshape(G * M, d)).reshape(G, M, d)
        S_next = _log_euler_step(S, sigma_n, L_n, xi_n, dt, r)

        corr = _mart_correction_exact(phi_n, S, S_next, n, dt, r)
        pen = _penalty_step(rho_std_n, d, rho_min, rho_max, beta)

        mart = mart + corr
        if torch.is_tensor(pen):
            disc_n = math.exp(-r * n * dt)
            pen_sum = pen_sum + disc_n * dt * pen

        S = S_next

    return mart, pen_sum, S




def _warm_start_from_actors(actor_list, xi, x0, d, sigma_min, sigma_max, rho_min, rho_max, N, T, r, u_grid, device):
    M = xi.shape[0]
    dt = T / N
    n_ctrl = d * (d + 1) // 2

    dtp = xi.dtype
    u_opt = torch.empty(M, N, n_ctrl, device=device, dtype=dtp)
    S = x0.unsqueeze(0).expand(M, d).clone()

    for a in actor_list:
        a.to(device).eval()
        if a.input_mean is not None:
            a.input_mean = a.input_mean.to(device)
        if a.input_std is not None:
            a.input_std = a.input_std.to(device)

    with torch.inference_mode():
        for n in range(N):
            raw_latent = actor_list[n](S)
            u_n = torch.tanh(raw_latent)

            diffs = (u_n.unsqueeze(-1) - u_grid.view(1, 1, -1)).abs()
            best_idx = diffs.argmin(dim=-1)
            u_q = u_grid[best_idx]
            u_opt[:, n, :] = u_q

            sigma_n, L_n, _ = _controls_from_u(u_q, d, sigma_min, sigma_max, rho_min, rho_max)
            xi_n = xi[:, n, :]
            S = _log_euler_step(S, sigma_n, L_n, xi_n, dt, r)

    return u_opt




def compute_dual_bound_grid_coordinate(phi, g, x0, d, sigma_min, sigma_max, rho_min, rho_max, N, T, r, h, xi, n_sweeps, device, beta, actor_list):

    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    xi = xi.to(device)
    dtype = xi.dtype
    M = xi.shape[0]
    n_ctrl = d * (d + 1) // 2

    x0 = torch.as_tensor(x0, device=device, dtype=dtype)
    if x0.dim() == 0:
        x0 = x0.unsqueeze(0)

    sigma_min_t = torch.as_tensor(sigma_min, device=device, dtype=dtype)
    sigma_max_t = torch.as_tensor(sigma_max, device=device, dtype=dtype)

    if rho_min is not None:
        rho_min_t = torch.as_tensor(rho_min, device=device, dtype=dtype)
        rho_max_t = torch.as_tensor(rho_max, device=device, dtype=dtype)
    else:
        rho_min_t, rho_max_t = None, None

    u_grid = _u_hnet(h, device, dtype)
    G = u_grid.shape[0]

    if actor_list is not None:
        u_opt = _warm_start_from_actors(actor_list, xi, x0, d, sigma_min_t, sigma_max_t, rho_min_t, rho_max_t, N, T, r, u_grid, device)
    else:
        center_idx = G // 2
        u_opt = torch.full((M, N, n_ctrl), u_grid[center_idx].item(), device=device)

    states, mart_cum, pen_cum = _build_cache(u_opt, xi, phi, x0, d, sigma_min_t, sigma_max_t, rho_min_t, rho_max_t, N, T, r, beta)

    discount_T = math.exp(-r * T)

    for sw in range(n_sweeps):
        for n in range(N):
            S_n_expanded = states[:, n, :].unsqueeze(0).expand(G, -1, -1)

            for c in range(n_ctrl):
                cand_u_n = u_opt[:, n, :].unsqueeze(0).expand(G, -1, -1).clone()
                cand_u_n[:, :, c] = u_grid.view(G, 1)

                mart_suf, pen_suf, S_final = _forward_partial(n, S_n_expanded, cand_u_n, u_opt, xi, phi, d, sigma_min_t, sigma_max_t, rho_min_t, rho_max_t,
                                                              N, T, r, beta)

                terminal = discount_T * g(S_final.reshape(G * M, d)).view(G, M)

                value = (terminal - mart_cum[:, n].unsqueeze(0) - mart_suf - pen_cum[:, n].unsqueeze(0) - pen_suf)

                best_idx = value.argmax(dim=0)
                u_opt[:, n, c] = u_grid[best_idx]

            _update_cache_from(n, states, mart_cum, pen_cum, u_opt, xi, phi, d, sigma_min_t, sigma_max_t, rho_min_t, rho_max_t, N, T, r, beta)

    terminal_final = discount_T * g(states[:, N, :])
    value_final = terminal_final - mart_cum[:, N] - pen_cum[:, N]

    dual_price = value_final.mean().item()
    dual_std = value_final.std(unbiased=True).item()
    ci_half = 1.96 * dual_std / math.sqrt(M)

    return dual_price, ci_half, u_opt


## Experiment configuration

All settings below come from the original supplied implementation, except the requested architecture, learning-rate and Sobolev-weight adjustments. No seed is set.

In [ ]:
d = 3  # Choose 2, 3, 4 or 5. This is the only dimension switch.
if not (2 <= d <= 5):
    raise ValueError("The uncertain-correlation implementation supports 2 <= d <= 5.")

n_rho = d * (d - 1) // 2
T, N_dual = 1.0, 128
M_train, M_eval = 2**16, 2**19
I_outer, I_inner = 500, 10
batch_size = 2**11
r = 0.0
S_0 = torch.full((d,), 100.0, dtype=torch.float32)
beta = 10.0
eta_V, eta_dV = 1.0, 0.1

sigma_min = torch.full((d,), 0.1)
sigma_max = torch.full((d,), 0.2)
rho_min = torch.full((n_rho,), -0.5)
rho_max = torch.full((n_rho,), 0.5)
lbd_max = torch.cat((torch.full((d,), 1.0), torch.full((n_rho,), 1.0)))
lbd_min = torch.cat((torch.full((d,), 0.01), torch.full((n_rho,), 0.01)))
epsilon_clip = 0.2

critic_arch = dict(input_dim=d, hidden_dim=64)
sobolev_critic_arch = dict(input_dim=d, hidden_dim=512)
actor_arch = dict(
    input_dim=d, hidden_dim=32, output_dim=d + n_rho, d=d,
    sigma_min=sigma_min, sigma_max=sigma_max,
    rho_min=rho_min, rho_max=rho_max,
)

critic_lr_max_1 = critic_lr_min_1 = critic_lr_max_2 = critic_lr_min_2 = 1e-3
actor_lr_max_1 = actor_lr_min_1 = actor_lr_max_2 = actor_lr_min_2 = 1e-3

M_dual = 2**10
h = 0.1
n_sw = 3
beta_dual = 1e5

def g(s):
    basket = torch.exp(torch.log(s[:, 1:]).mean(dim=1))
    return torch.clamp(basket - s[:, 0], min=0.0)

def running_reward(s, action, t):
    return torch.zeros(s.shape[0], device=s.device, dtype=s.dtype)

print(f"Option: geo-outperformer | d={d} | controls={d + n_rho}")


## 1. Classical SPG

Train the classical actor and critic backward in time, then evaluate the learned deterministic actors on independent paths.

In [ ]:
classical_critics, classical_actors = train_all_classical(
    T, N_dual, M_train, S_0, r,
    critic_arch, actor_arch, batch_size,
    I_outer, I_inner, sigma_min, sigma_max,
    rho_min, rho_max, lbd_max, lbd_min, epsilon_clip,
    critic_lr_max_1, critic_lr_max_2,
    critic_lr_min_1, critic_lr_min_2,
    actor_lr_max_1, actor_lr_max_2,
    actor_lr_min_1, actor_lr_min_2,
    beta, running_reward, g,
)

classical_price, classical_ci_low, classical_ci_high = price_from_actors(
    classical_actors, S_0, r, T, N_dual, M_eval,
    running_reward, g, rho_min=rho_min, rho_max=rho_max,
)
classical_critic_price = classical_critics[0](S_0.to(device).unsqueeze(0)).item()
print(f"Classical primal: {classical_price.item():.6f} "
      f"(95% CI [{classical_ci_low.item():.6f}, {classical_ci_high.item():.6f}])")
print(f"Classical initial critic: {classical_critic_price:.6f}")


## 2. Sobolev SPG

Start a fresh and independent backward training run with the Sobolev critic.

In [ ]:
sobolev_critics, sobolev_actors = train_all_sobolev(
    T, N_dual, M_train, S_0, r,
    sobolev_critic_arch, actor_arch, batch_size,
    I_outer, I_inner, sigma_min, sigma_max,
    rho_min, rho_max, lbd_max, lbd_min, epsilon_clip,
    critic_lr_max_1, critic_lr_max_2,
    critic_lr_min_1, critic_lr_min_2,
    actor_lr_max_1, actor_lr_max_2,
    actor_lr_min_1, actor_lr_min_2,
    eta_V, eta_dV, beta, running_reward, g,
)

sobolev_price, sobolev_ci_low, sobolev_ci_high = price_from_actors(
    sobolev_actors, S_0, r, T, N_dual, M_eval,
    running_reward, g, rho_min=rho_min, rho_max=rho_max,
)
sobolev_critic_price = sobolev_critics[0](S_0.to(device).unsqueeze(0)).item()
print(f"Sobolev primal: {sobolev_price.item():.6f} "
      f"(95% CI [{sobolev_ci_low.item():.6f}, {sobolev_ci_high.item():.6f}])")
print(f"Sobolev initial critic: {sobolev_critic_price:.6f}")


## 3. Critic gradients for the dual formulation

The gradients are computed through autograd with respect to the physical state. No finite-difference reference is used.

In [ ]:
def make_delta_from_critics(critic_list, d, device):
    for net in critic_list:
        net.to(device).eval()
        for p in net.parameters():
            p.requires_grad_(False)
        if torch.is_tensor(net.input_mean):
            net.input_mean = net.input_mean.to(device)
        if torch.is_tensor(net.input_std):
            net.input_std = net.input_std.to(device)

    def delta(k, x):
        x_in = x.reshape(-1, d).detach().to(device=device, dtype=torch.float32)
        x_in.requires_grad_(True)
        with torch.enable_grad():
            u = critic_list[k](x_in)
            (du_dx,) = torch.autograd.grad(u.sum(), x_in, create_graph=False)
        return du_dx.detach()
    return delta


delta_cl = make_delta_from_critics(classical_critics, d, device)
delta_so = make_delta_from_critics(sobolev_critics, d, device)

# Use identical Brownian increments in the two dual comparisons.
xi = torch.randn(M_dual, N_dual, d, device=device)


## 4. Classical-gradient dual estimate

The actor initializes the pathwise grid search. This finite-sweep estimate is not certified.

In [ ]:
res_cl = compute_dual_bound_grid_coordinate(
    delta_cl, g, S_0, d, sigma_min, sigma_max,
    rho_min, rho_max, N_dual, T, r,
    h, xi, n_sw, device, beta_dual, classical_actors,
)
print(f"Classical-gradient dual estimate: {res_cl[0]:.6f} +/- {res_cl[1]:.6f} (95% MC half-width)")


## 5. Sobolev-gradient dual estimate

The same Brownian increments are used for the Sobolev dual evaluation.

In [ ]:
res_so = compute_dual_bound_grid_coordinate(
    delta_so, g, S_0, d, sigma_min, sigma_max,
    rho_min, rho_max, N_dual, T, r,
    h, xi, n_sw, device, beta_dual, sobolev_actors,
)
print(f"Sobolev-gradient dual estimate: {res_so[0]:.6f} +/- {res_so[1]:.6f} (95% MC half-width)")


## 6. Comparison

Primal intervals are independent Monte Carlo 95% confidence intervals. Dual error bars represent only the Monte Carlo uncertainty of the approximate pathwise optimization, not its optimization or discretization bias.

In [ ]:
print(f"{'Method':<22} {'Primal':>12} {'Critic':>12} {'Dual estimate':>16}")
print("-" * 66)
print(f"{'Classical SPG':<22} {classical_price.item():>12.4f} {classical_critic_price:>12.4f} {res_cl[0]:>16.4f}")
print(f"{'Sobolev SPG':<22} {sobolev_price.item():>12.4f} {sobolev_critic_price:>12.4f} {res_so[0]:>16.4f}")

labels = ["Classical primal", "Sobolev primal", "Classical dual", "Sobolev dual"]
values = [classical_price.item(), sobolev_price.item(), res_cl[0], res_so[0]]
errors = [0.5 * (classical_ci_high - classical_ci_low).item(),
          0.5 * (sobolev_ci_high - sobolev_ci_low).item(), res_cl[1], res_so[1]]
fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(range(4), values, yerr=errors, fmt="o", capsize=4)
ax.set_xticks(range(4), labels, rotation=15, ha="right")
ax.set_ylabel("Price estimate")
ax.set_title("Primal and approximate dual estimates")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()